# Vivir más sin ser más rico

**Trabajo Práctico Final · Visualización de la Información · ITBA**

- Javier Peral Belmont
- Pilar Frutos

**Fecha de entrega:** noviembre de 2026

¿Cuánto se parecen la riqueza de un país y los años que vive su gente? Exploramos la API de **Data360** del Banco Mundial y contamos la historia en tres visualizaciones interactivas:

1. **PIB por habitante y esperanza de vida (2024).** Los países más ricos suelen vivir más, pero algunos viven bastante más (o menos) de lo que les correspondería por su ingreso.
2. **Acceso básico al agua para beber y esperanza de vida (2000–2024).** Un servicio esencial, cómo se relaciona con la longevidad y cómo cambió esa relación.
3. **Vacunación infantil contra el sarampión y esperanza de vida (2000–2024).** Una medida de prevención que plantea una pregunta distinta a la del agua.

**Cómo ejecutarlo:** *Entorno de ejecución → Ejecutar todo*. Los datos se descargan de la API al correr el notebook (tarda alrededor de un minuto) y los gráficos se arman con Altair en este mismo notebook.

In [ ]:
!pip -q install babel pycountry

import time

import altair as alt
import numpy as np
import pandas as pd
import pycountry
import requests
from babel import Locale

alt.data_transformers.disable_max_rows()
alt.renderers.set_embed_options(format_locale="es-ES")   # coma decimal en ejes y fichas

API = "https://data360api.worldbank.org/data360"
DESDE, HASTA = 2000, 2024


In [2]:
def pedir(metodo, ruta, intentos=6, **kwargs):
    """Llama a la API reintentando: a veces responde con el cuerpo vacío o corta la conexión."""
    for intento in range(intentos):
        try:
            r = requests.request(metodo, f"{API}/{ruta}", timeout=120, **kwargs)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if intento == intentos - 1:
                raise
            time.sleep(2 * (intento + 1))


# 1. Exploración de Data360

La plataforma Data360 del Banco Mundial reúne indicadores del propio Banco y de otros organismos internacionales (ONU, OMS, UNESCO, FMI, entre otros). Antes de elegir los indicadores, recorremos el catálogo a través de la API.

### Bases de datos disponibles

El endpoint de búsqueda (`/searchv2`) permite pedir *facetas*: cuántos indicadores hay en cada base de datos.

In [3]:
respuesta = pedir("POST", "searchv2", json={
    "search": "*",
    "top": 0,
    "count": True,
    "filter": "type eq 'indicator'",
    "facets": ["series_description/database_id,count:1000"],
})

bases = (pd.DataFrame(respuesta["@search.facets"]["series_description/database_id"])
         .rename(columns={"value": "base", "count": "indicadores"})
         .sort_values("indicadores", ascending=False, ignore_index=True))

print("Bases de datos en Data360:", len(bases))
print("Indicadores en total:", bases["indicadores"].sum())
bases.head(15)


Bases de datos en Data360: 171
Indicadores en total: 9850


,base,indicadores
0,WB_WDI,1545
1,IMF_BOP,1146
2,WB_EDSTATS,1072
3,WB_ES,584
4,IMF_IFS,495
5,WB_GS,363
6,WB_FINDEX,280
7,IMF_GFS_SOO,218
8,BS_SGI,193
9,WB_HNP,192


In [4]:
alt.Chart(bases.head(20)).mark_bar(color="#2a78d6", cornerRadiusEnd=3).encode(
    x=alt.X("indicadores:Q", title="Cantidad de indicadores"),
    y=alt.Y("base:N", sort="-x", title=None),
    tooltip=[alt.Tooltip("base:N", title="Base"), alt.Tooltip("indicadores:Q", title="Indicadores")],
).properties(width=600, height=420, title="Las 20 bases de Data360 con más indicadores")


alt.Chart(...)

Elegimos **World Development Indicators** (`WB_WDI`): es la base con más indicadores, tiene series anuales por país desde 1960 y cubre economía, salud, educación e infraestructura, que es lo que necesitamos para cruzar variables.

Con el endpoint `/indicators` obtenemos la lista completa de sus indicadores. Cada código de WDI lleva adelante el nombre de la base: `SP.DYN.LE00.IN` pasa a ser `WB_WDI_SP_DYN_LE00_IN`.

In [5]:
indicadores_wdi = pd.DataFrame(pedir("GET", "indicators", params={"datasetId": "WB_WDI"}), columns=["indicator_id"])
print("Indicadores en WB_WDI:", len(indicadores_wdi))

# El tercer bloque del código indica el área temática (SP: población, SH: salud, NY: cuentas nacionales…)
indicadores_wdi["tema"] = indicadores_wdi["indicator_id"].str.split("_").str[2]
indicadores_wdi["tema"].value_counts().head(12)


Indicadores en WB_WDI: 1499


tema
SH    158
SE    156
SL    129
SP     98
NY     94
DT     84
IC     77
NE     65
EN     63
GC     45
TM     45
DC     38
Name: count, dtype: int64

In [6]:
indicadores_wdi.sample(12, random_state=1)

,indicator_id,tema
91,WB_WDI_SE_ENR_PRSC_FM_ZS,SE
75,WB_WDI_NV_MNF_FBTO_ZS_UN,NV
1211,WB_WDI_NY_GNP_MKTP_KD,NY
330,WB_WDI_SL_TLF_ACTI_MA_ZS,SL
961,WB_WDI_FX_OWN_TOTL_MA_ZS,FX
880,WB_WDI_SI_DST_03RD_20,SI
930,WB_WDI_DC_DAC_CZEL_CD,DC
1076,WB_WDI_SL_IND_EMPL_ZS,SL
1382,WB_WDI_IP_PAT_RESD,IP
335,WB_WDI_SP_M15_2024_FE_ZS,SP


### ¿Dónde está la esperanza de vida?

Buscamos «life expectancy at birth» en todo el catálogo. El mismo dato aparece en varias bases, así que conviene elegir una y usarla para todo.

In [7]:
busqueda = pedir("POST", "searchv2", json={
    "search": "life expectancy at birth",
    "top": 15,
    "select": "series_description/idno, series_description/name, series_description/database_id",
})
pd.DataFrame([v["series_description"] for v in busqueda["value"]])


,idno,name,database_id
0,WB_WDI_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)",WB_WDI
1,WB_WDI_SP_DYN_LE00_FE_IN,"Life expectancy at birth, female (years)",WB_WDI
2,WB_WDI_SP_DYN_LE00_MA_IN,"Life expectancy at birth, male (years)",WB_WDI
3,WB_HNP_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)",WB_HNP
4,WB_GS_SP_DYN_LE00_IN,Life expectancy at birth (years),WB_GS
5,WB_ESG_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)",WB_ESG
6,WB_HCP_SP_DYN_LE00_IN,Life expectancy at birth (years),WB_HCP
7,WB_GS_SP_DYN_LE60_IN,Life expectancy at age 60 (years),WB_GS
8,WB_WDI_SP_POP_BRTH_MF,Sex ratio at birth (male births per female bir...,WB_WDI
9,WB_GS_SP_REG_BRTH_ZS,Completeness of birth registration (%),WB_GS


### Estructura de una observación

El endpoint `/data` devuelve las observaciones en formato SDMX, paginadas de a 1000 filas. Cada fila trae el país o región (`REF_AREA`), el año (`TIME_PERIOD`), el valor (`OBS_VALUE`, que llega como texto) y las dimensiones de desagregación (sexo, edad, urbanización), además de metadatos de la observación. Este es el dato de Argentina en 2024:

In [8]:
obs = pedir("GET", "data", params={"DATABASE_ID": "WB_WDI", "INDICATOR": "WB_WDI_SP_DYN_LE00_IN",
                                   "REF_AREA": "ARG", "TIME_PERIOD": 2024})["value"][0]
pd.DataFrame([obs]).T


,0
OBS_VALUE,77.543
TIME_FORMAT,P1Y
UNIT_MULT,0
COMMENT_OBS,None
OBS_STATUS,A
OBS_CONF,PU
AGG_METHOD,_Z
DECIMALS,2
COMMENT_TS,None
DATA_SOURCE,None


# 2. Selección de indicadores

Para cada gráfico necesitamos series históricas que se puedan cruzar por código de país y año. Además de la esperanza de vida, el PIB y la población, evaluamos los tres indicadores propuestos y tres alternativas para el tercer gráfico. Consultamos la ficha de cada uno en el endpoint `/metadata` para respetar su nombre y su unidad.

In [9]:
INDICADORES = {
    # clave: (código Data360, nombre en castellano, unidad)
    "vida": ("WB_WDI_SP_DYN_LE00_IN", "Esperanza de vida al nacer", "años"),
    "pib": ("WB_WDI_NY_GDP_PCAP_PP_KD", "PIB per cápita en paridad de poder adquisitivo", "$ internacionales constantes de 2021"),
    "poblacion": ("WB_WDI_SP_POP_TOTL", "Población total", "personas"),
    "agua": ("WB_WDI_SH_H2O_BASW_ZS", "Población que usa al menos un servicio básico de agua para beber", "% de la población"),
    "secundaria": ("WB_WDI_SE_SEC_CMPT_LO_ZS", "Tasa de finalización del primer ciclo de secundaria", "% del grupo de edad correspondiente"),
    "sarampion": ("WB_WDI_SH_IMM_MEAS", "Inmunización contra el sarampión (primera dosis)", "% de niños de 12 a 23 meses"),
    "dpt": ("WB_WDI_SH_IMM_IDPT", "Inmunización contra difteria, tos convulsa y tétanos (DPT)", "% de niños de 12 a 23 meses"),
    "saneamiento": ("WB_WDI_SH_STA_BASS_ZS", "Población que usa al menos servicios básicos de saneamiento", "% de la población"),
    "uhc": ("WB_WDI_SH_UHC_SCI", "Índice de cobertura de servicios de salud (UHC)", "índice de 0 a 100"),
}

fichas = []
for clave, (codigo, nombre, unidad) in INDICADORES.items():
    meta = pedir("POST", "metadata", json={"query": f"&$filter=series_description/idno eq '{codigo}'"})
    sd = meta["value"][0]["series_description"]
    fichas.append({"clave": clave, "código": codigo, "nombre original": sd["name"],
                   "unidad original": sd.get("measurement_unit"), "nombre en el notebook": nombre, "unidad": unidad})
pd.DataFrame(fichas).set_index("clave")


,código,nombre original,unidad original,nombre en el notebook,unidad
clave,,,,,
vida,WB_WDI_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)",Years,Esperanza de vida al nacer,años
pib,WB_WDI_NY_GDP_PCAP_PP_KD,"GDP per capita, PPP (constant 2021 internation...",Constant 2021 international $,PIB per cápita en paridad de poder adquisitivo,$ internacionales constantes de 2021
poblacion,WB_WDI_SP_POP_TOTL,"Population, total",Unit,Población total,personas
agua,WB_WDI_SH_H2O_BASW_ZS,People using at least basic drinking water ser...,% of population,Población que usa al menos un servicio básico ...,% de la población
secundaria,WB_WDI_SE_SEC_CMPT_LO_ZS,"Lower secondary completion rate, total (% of r...",% of relevant age group,Tasa de finalización del primer ciclo de secun...,% del grupo de edad correspondiente
sarampion,WB_WDI_SH_IMM_MEAS,"Immunization, measles (% of children ages 12-2...",Percentage,Inmunización contra el sarampión (primera dosis),% de niños de 12 a 23 meses
dpt,WB_WDI_SH_IMM_IDPT,"Immunization, DPT (% of children ages 12-23 mo...",Percentage,"Inmunización contra difteria, tos convulsa y t...",% de niños de 12 a 23 meses
saneamiento,WB_WDI_SH_STA_BASS_ZS,People using at least basic sanitation service...,% of population,Población que usa al menos servicios básicos d...,% de la población
uhc,WB_WDI_SH_UHC_SCI,UHC service coverage index,Index,Índice de cobertura de servicios de salud (UHC),índice de 0 a 100


Dos detalles de las fichas que condicionan la lectura:

- El indicador de agua mide **porcentaje de la población** que usa al menos un servicio básico: una fuente mejorada a no más de 30 minutos, ida y vuelta. No mide la calidad del agua (para eso existe otra serie, la de agua «gestionada de forma segura»).
- La finalización de secundaria es una tasa **bruta**: cuenta a todos los que llegan al último grado, tengan la edad que tengan, así que puede superar el 100 %.

Descargamos todas las observaciones de cada indicador:

In [10]:
def traer_indicador(codigo, columna):
    """Descarga todas las páginas de un indicador y devuelve una tabla país–año."""
    filas, skip = [], 0
    while True:
        pagina = pedir("GET", "data", params={"DATABASE_ID": "WB_WDI", "INDICATOR": codigo, "top": 1000, "skip": skip})
        filas += pagina["value"]
        skip += 1000
        if skip >= pagina["count"] or not pagina["value"]:
            break
    d = pd.DataFrame(filas)
    for dim in ["SEX", "AGE", "URBANISATION"]:      # nos quedamos con el total si hay desagregaciones
        if dim in d and d[dim].nunique() > 1:
            d = d[d[dim] == "_T"]
    return pd.DataFrame({
        "iso3": d["REF_AREA"],
        "anio": pd.to_numeric(d["TIME_PERIOD"]).astype(int),
        columna: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    }).dropna()


series = {}
for clave, (codigo, _, _) in INDICADORES.items():
    series[clave] = traer_indicador(codigo, clave)
    s = series[clave]
    print(f"{clave:12} {len(s):6} filas  {s['anio'].min()}–{s['anio'].max()}  {s['iso3'].nunique()} códigos de país o región")


vida          17191 filas  1960–2024  265 códigos de país o región


pib            8705 filas  1990–2025  247 códigos de país o región


poblacion     17460 filas  1960–2025  265 códigos de país o región


agua           6398 filas  2000–2024  262 códigos de país o región


secundaria     5939 filas  1970–2025  242 códigos de país o región


sarampion     10046 filas  1980–2024  241 códigos de país o región


dpt           10169 filas  1980–2024  241 códigos de país o región


saneamiento    6363 filas  2000–2024  262 códigos de país o región


uhc            5784 filas  2000–2023  241 códigos de país o región


### Países, no agregados

La API mezcla países con agregados como `WLD` (mundo), `LCN` (América Latina y el Caribe) o `HIC` (ingreso alto). Para que un agregado no cuente como un punto más, nos quedamos solo con las economías de la jerarquía de regiones del Banco Mundial (`/portal/v1/hierarchy`). De ahí sacamos también la región y el grupo de ingreso, y con `babel` y `pycountry` ponemos los nombres en castellano.

In [11]:
jerarquia = pedir("GET", "portal/v1/hierarchy", params={"type": "country"})["data"]

REGIONES = {
    "East Asia & Pacific": "Asia oriental y Pacífico",
    "Europe & Central Asia": "Europa y Asia central",
    "Latin America & Caribbean": "América Latina y el Caribe",
    "Middle East, North Africa, Afghanistan & Pakistan": "Medio Oriente y norte de África",
    "North America": "América del Norte",
    "South Asia": "Asia meridional",
    "Eastern and Southern Africa": "África subsahariana",
    "Western and Central Africa": "África subsahariana",
}
INGRESOS = {"Low income": "Bajo", "Lower middle income": "Medio bajo", "Upper middle income": "Medio alto", "High income": "Alto"}

regiones = pd.DataFrame([{"iso3": c["id"], "nombre_en": c["name"], "region": REGIONES[g["name"]]}
                         for g in jerarquia["HCL_WB_REGIONS"] for c in g["countries"]])
ingresos = pd.DataFrame([{"iso3": c["id"], "ingreso": INGRESOS.get(g["name"])}
                         for g in jerarquia["HCL_WB_INCOME"] for c in g["countries"]])
paises = regiones.merge(ingresos, on="iso3", how="left")
paises["ingreso"] = paises["ingreso"].fillna("Sin clasificar")

nombres_es = Locale("es").territories
def nombre_es(fila):
    pais = pycountry.countries.get(alpha_3=fila.iso3)
    iso2 = pais.alpha_2 if pais else {"XKX": "XK"}.get(fila.iso3)
    return nombres_es.get(iso2, {"CHI": "Islas del Canal"}.get(fila.iso3, fila.nombre_en))

paises["pais"] = paises.apply(nombre_es, axis=1)

codigos = set(series["vida"]["iso3"])
agregados = sorted(codigos - set(paises["iso3"]))
print(f"{len(paises)} economías en la jerarquía · {len(codigos)} códigos con esperanza de vida · {len(agregados)} agregados excluidos")
print("Algunos agregados:", ", ".join(agregados[:20]), "…")
paises.sample(6, random_state=3)


220 economías en la jerarquía · 265 códigos con esperanza de vida · 48 agregados excluidos
Algunos agregados: AFE, AFW, ARB, CEB, CSS, EAP, EAR, EAS, ECA, ECS, EMU, EUU, FCV, HIC, HPC, IBD, IBT, IDA, IDB, IDX …


,iso3,nombre_en,region,ingreso,pais
130,RUS,Russian Federation,Europa y Asia central,Alto,Rusia
64,MYS,Malaysia,Asia oriental y Pacífico,Medio alto,Malasia
154,CHL,Chile,América Latina y el Caribe,Alto,Chile
86,ALB,Albania,Europa y Asia central,Medio alto,Albania
70,NCL,New Caledonia,Asia oriental y Pacífico,Alto,Nueva Caledonia
135,ESP,Spain,Europa y Asia central,Alto,España


Unificamos las dos regiones africanas del Banco Mundial en «África subsahariana». Con todo esto armamos un panel país–año desde 1990:

In [12]:
panel = None
for clave, s in series.items():
    panel = s if panel is None else panel.merge(s, on=["iso3", "anio"], how="outer")

panel = (panel[panel["anio"].between(1990, HASTA)]
         .merge(paises[["iso3", "pais", "region", "ingreso"]], on="iso3", how="inner")
         .sort_values(["iso3", "anio"], ignore_index=True))
print(panel.shape)
panel.sample(5, random_state=2)


(7595, 14)


,iso3,anio,vida,pib,poblacion,agua,secundaria,sarampion,dpt,saneamiento,uhc,pais,region,ingreso
1709,CYP,2019,81.454000,45454.175781,1286671,99.753710,98.962860,86.0,96.0,99.502639,76.0,Chipre,Europa y Asia central,Alto
6773,TLS,2008,62.858000,3530.882175,1032429,59.681726,56.217232,73.0,79.0,33.556386,36.0,Timor-Leste,Asia oriental y Pacífico,Medio bajo
4817,MWI,2012,59.870000,1518.148038,15709577,63.625041,19.313887,90.0,96.0,27.422602,45.0,Malaui,África subsahariana,Bajo
7307,VIR,2017,79.368293,43277.154652,107281,99.245617,NaN,NaN,NaN,99.209037,NaN,Islas Vírgenes de EE. UU.,América Latina y el Caribe,Alto
6925,TUR,2020,76.525000,28976.717916,83384680,95.952576,131.028015,95.0,98.0,99.230742,77.0,Turquía,Europa y Asia central,Medio alto


# 3. Cobertura, faltantes y período

### ¿Cuántos países tienen los dos datos en cada año?

Para cada candidato contamos, año por año, cuántos países tienen a la vez ese indicador y la esperanza de vida. Además medimos si la composición de la muestra cambia:

- **% población:** qué parte de la población de todas las economías del panel representan esos países.
- **Coincidencia con 2015:** índice de Jaccard entre los países del año y los de 2015 (países en ambos ÷ países en alguno). Vale 1 si son exactamente los mismos y baja a medida que la muestra cambia.

In [13]:
CANDIDATOS = {"agua": "Agua básica", "secundaria": "Secundaria", "sarampion": "Sarampión",
              "dpt": "DPT", "saneamiento": "Saneamiento", "uhc": "Índice UHC"}
poblacion_total = panel.groupby("anio")["poblacion"].sum()

def cobertura(col, referencia=2015):
    con = panel[panel[col].notna() & panel["vida"].notna()]
    conjuntos = con.groupby("anio")["iso3"].agg(set)
    base = conjuntos.get(referencia, set())
    return pd.DataFrame({
        "países": con.groupby("anio").size(),
        "% población": (con.groupby("anio")["poblacion"].sum() / poblacion_total * 100).round(1),
        "coincidencia con 2015": conjuntos.map(lambda s: len(s & base) / len(s | base)).round(2),
    }).dropna(subset=["países"])

coberturas = {c: cobertura(c) for c in CANDIDATOS}
tabla = pd.concat(coberturas, axis=1).reindex(range(1990, HASTA + 1))
tabla.loc[[1990, 1995, 2000, 2005, 2010, 2015, 2019, 2020, 2022, 2023, 2024]]


agua                                   secundaria              \
     países % población coincidencia con 2015     países % población   
anio                                                                   
1990    NaN         NaN                   NaN         43        31.1   
1995    NaN         NaN                   NaN         36        29.7   
2000  203.0        93.7                  0.95         76        20.4   
2005  209.0        99.7                  0.98        113        34.3   
2010  212.0        99.7                  1.00        128        54.2   
2015  213.0        99.9                  1.00        126        51.8   
2019  206.0        99.2                  0.96        137        58.3   
2020  205.0        99.2                  0.95        128        52.7   
2022  203.0        99.2                  0.94        135        59.8   
2023  203.0        99.2                  0.94        135        57.7   
2024  202.0        98.8                  0.94        108        48.3   

                           sarampion                                      dpt  \
     coincidencia con 2015    países % población coincidencia con 2015 países   
anio                                                                            
1990                  0.22       160        93.3                  0.83    162   
1995                  0.21       186        99.5                  0.96    187   
2000                  0.32       190        99.6                  0.98    190   
2005                  0.53       191        99.7                  0.99    191   
2010                  0.63       192        99.6                  0.99    192   
2015                  1.00       193        99.8                  1.00    193   
2019                  0.66       193        99.8                  1.00    193   
2020                  0.57       193        99.8                  1.00    193   
2022                  0.59       193        99.8                  1.00    193   
2023                  0.58       193        99.8                  1.00    193   
2024                  0.50       192        99.7                  0.99    192   

                                       saneamiento              \
     % población coincidencia con 2015      países % población   
anio                                                             
1990        93.3                  0.84         NaN         NaN   
1995        99.5                  0.97         NaN         NaN   
2000        99.6                  0.98       202.0        96.3   
2005        99.7                  0.99       208.0        99.4   
2010        99.6                  0.99       210.0        99.4   
2015        99.8                  1.00       213.0        99.9   
2019        99.8                  1.00       203.0        99.2   
2020        99.8                  1.00       202.0        99.2   
2022        99.8                  1.00       199.0        99.1   
2023        99.8                  1.00       196.0        98.6   
2024        99.7                  0.99       193.0        98.1   

                              uhc                                    
     coincidencia con 2015 países % población coincidencia con 2015  
anio                                                                 
1990                   NaN    NaN         NaN                   NaN  
1995                   NaN    NaN         NaN                   NaN  
2000                  0.95  193.0        99.8                   1.0  
2005                  0.98  193.0        99.8                   1.0  
2010                  0.99  193.0        99.8                   1.0  
2015                  1.00  193.0        99.8                   1.0  
2019                  0.94  193.0        99.8                   1.0  
2020                  0.94  193.0        99.8                   1.0  
2022                  0.93  193.0        99.8                   1.0  
2023                  0.91  193.0        99.8                   1.0  
2024                  0.90  

In [14]:
largo = pd.concat([c.reset_index().assign(indicador=CANDIDATOS[k]) for k, c in coberturas.items()])
base = alt.Chart(largo).encode(
    x=alt.X("anio:Q", title="Año", axis=alt.Axis(format="d")),
    color=alt.Color("indicador:N", title="Indicador",
                    scale=alt.Scale(domain=list(CANDIDATOS.values()),
                                    range=["#2a78d6", "#eb6834", "#1baf7a", "#9ec5f4", "#eda100", "#e87ba4"])),
    tooltip=["indicador:N", alt.Tooltip("anio:Q", title="Año", format="d"), "países:Q", "% población:Q", "coincidencia con 2015:Q"],
)
cantidad = base.mark_line(point=True, strokeWidth=2).encode(y=alt.Y("países:Q", title="Países con ambos datos")).properties(
    width=380, height=260, title="Países con el indicador y la esperanza de vida")
coincidencia = base.mark_line(point=True, strokeWidth=2).encode(
    y=alt.Y("coincidencia con 2015:Q", title="Coincidencia con 2015", scale=alt.Scale(domain=[0, 1]))).properties(
    width=380, height=260, title="¿Son los mismos países que en 2015?")
cantidad | coincidencia


alt.HConcatChart(...)

**Lectura.** Agua, sarampión, DPT y saneamiento tienen más de 190 países por año y prácticamente la misma muestra en todo el período: desde 2000, la coincidencia con 2015 no baja de 0,9. La secundaria es otra historia. Nunca supera los 140 países, en varios años cubre a menos de la mitad de la población y la muestra cambia mucho de un año a otro.

### ¿Alcanza la secundaria?

La consigna proponía la finalización de secundaria como primera opción para el tercer gráfico. Miramos qué países aparecen cada año:

In [15]:
sec = panel[panel["secundaria"].notna() & panel["vida"].notna() & panel["anio"].between(DESDE, HASTA)]
anios_por_pais = sec.groupby("iso3")["anio"].nunique()
print(f"Países con algún dato entre {DESDE} y {HASTA}: {anios_por_pais.size}")
print(f"Países con datos en al menos 20 de los 25 años: {(anios_por_pais >= 20).sum()}")
print(f"Observaciones por encima del 100 % (tasa bruta): {(sec['secundaria'] > 100).sum()} de {len(sec)}")
pd.crosstab(sec["region"], sec["anio"]).loc[:, [2005, 2010, 2015, 2020, 2023, 2024]]


Países con algún dato entre 2000 y 2024: 192
Países con datos en al menos 20 de los 25 años: 62
Observaciones por encima del 100 % (tasa bruta): 393 de 3021


anio,2005,2010,2015,2020,2023,2024
region,,,,,,
América Latina y el Caribe,28,28,24,22,25,22
América del Norte,1,0,1,1,0,0
Asia meridional,3,5,4,4,6,5
Asia oriental y Pacífico,13,15,14,22,24,22
Europa y Asia central,29,39,43,46,45,32
Medio Oriente y norte de África,14,14,11,14,15,11
África subsahariana,25,27,29,19,20,16


Con la secundaria, cada año del deslizador mostraría un conjunto distinto de países: algunas regiones aparecen y desaparecen, y pocos países tienen la serie casi completa. Un cambio en la nube de puntos podría deberse a que cambió la muestra y no la relación. Por eso **descartamos la secundaria** para el tercer gráfico.

Entre las alternativas:

- **Saneamiento básico** tiene muy buena cobertura, pero responde casi la misma pregunta que el agua: viene del mismo programa de monitoreo de agua y saneamiento de la OMS y UNICEF, y en 2024 su correlación con el agua es de 0,9 (ver abajo).
- **El índice UHC** cubre 2000–2023 y muchos países, pero es un índice compuesto de 14 indicadores trazadores (salud materno-infantil, enfermedades infecciosas, enfermedades no transmisibles y capacidad de los servicios). Al mezclar tantas cosas cuesta saber qué mide cada punto, y no llega a 2024.
- **La vacunación contra el sarampión** tiene la mejor combinación. Hay más de 190 países por año desde 2000, la muestra casi no cambia y plantea una pregunta distinta a la del agua: ya no es infraestructura, sino prevención en la primera infancia. Además, tiene una referencia clara, porque la OMS considera que hace falta una cobertura de al menos 95 % para frenar la circulación del virus. La DPT tiene una cobertura casi idéntica, pero sin un umbral tan conocido.

Elegimos **sarampión, primera dosis**.

In [16]:
for clave in ["agua", "sarampion", "saneamiento"]:
    c = coberturas[clave].loc[DESDE:HASTA]
    print(f"{CANDIDATOS[clave]:12} {DESDE}–{HASTA}: entre {c['países'].min():.0f} y {c['países'].max():.0f} países por año, "
          f"coincidencia mínima con 2015: {c['coincidencia con 2015'].min():.2f}")

ultimo = panel[panel["anio"] == HASTA]
print(f"\nCorrelación entre agua y saneamiento en {HASTA}: {ultimo['agua'].corr(ultimo['saneamiento']):.2f}")


Agua básica  2000–2024: entre 202 y 214 países por año, coincidencia mínima con 2015: 0.94
Sarampión    2000–2024: entre 190 y 193 países por año, coincidencia mínima con 2015: 0.98
Saneamiento  2000–2024: entre 193 y 213 países por año, coincidencia mínima con 2015: 0.90

Correlación entre agua y saneamiento en 2024: 0.90


### Período y faltantes

**Período: 2000–2024 para los dos gráficos con deslizador.** La serie de agua empieza en 2000 y 2024 es el último año con esperanza de vida. La vacunación tiene datos desde antes, pero usamos la misma ventana para poder comparar los dos gráficos.

**Faltantes.** No imputamos ni interpolamos nada: en cada año se muestran solo los países que tienen los dos datos en ese año. Estos son los países con huecos:

In [17]:
def huecos(col):
    con = panel[panel["anio"].between(DESDE, HASTA) & panel[col].notna() & panel["vida"].notna()]
    resumen = con.groupby(["iso3", "pais"])["anio"].agg(primero="min", ultimo="max", anios="nunique").reset_index()
    return resumen[resumen["anios"] < HASTA - DESDE + 1].sort_values("ultimo")

print("Agua: países sin la serie completa", DESDE, "–", HASTA)
display(huecos("agua"))
print("Sarampión: países sin la serie completa")
display(huecos("sarampion"))


Agua: países sin la serie completa 2000 – 2024


,iso3,pais,primero,ultimo,anios
0,ABW,Aruba,2000,2016,17
6,ARG,Argentina,2000,2016,17
57,ERI,Eritrea,2000,2016,17
182,SXM,Sint Maarten,2000,2017,18
103,KNA,San Cristóbal y Nieves,2000,2017,18
76,GRD,Granada,2000,2017,18
74,GNQ,Guinea Ecuatorial,2000,2017,18
203,VCT,San Vicente y las Granadinas,2000,2018,19
20,BHS,Bahamas,2000,2019,20
65,FSM,Micronesia,2000,2020,21


Sarampión: países sin la serie completa


,iso3,pais,primero,ultimo,anios
126,NIC,Nicaragua,2000,2023,24
116,MNE,Montenegro,2006,2024,19
158,SSD,Sudán del Sur,2011,2024,14
172,TLS,Timor-Leste,2002,2024,23


En agua, la serie se corta antes de 2024 en 12 países, entre ellos **Argentina** (último dato en 2016), Eritrea, Guinea Ecuatorial y Congo. En otros 11 empieza más tarde: Estados Unidos y Polonia en 2005, y Sudán del Sur en 2011, cuando se independizó. En vacunación hay solo cuatro casos. Nicaragua no tiene dato de 2024, y Montenegro, Sudán del Sur y Timor-Leste empiezan después de 2000, cuando se convirtieron en países independientes.

¿Estos huecos cambian la relación? Comparamos la correlación de cada año con la que se obtiene usando solo los países con la serie completa (panel balanceado):

In [18]:
def correlaciones(col):
    con = panel[panel["anio"].between(DESDE, HASTA) & panel[col].notna() & panel["vida"].notna()]
    completos = con.groupby("iso3")["anio"].nunique().pipe(lambda s: s[s == HASTA - DESDE + 1].index)
    todos = con.groupby("anio")[[col, "vida"]].corr().xs(col, level=1)["vida"]
    balanceado = con[con["iso3"].isin(completos)].groupby("anio")[[col, "vida"]].corr().xs(col, level=1)["vida"]
    return pd.DataFrame({"todos los países": todos, f"solo los {len(completos)} completos": balanceado}).round(2)

pd.concat({"agua": correlaciones("agua"), "sarampión": correlaciones("sarampion")}, axis=1).loc[[2000, 2005, 2010, 2015, 2020, 2024]]


agua                               sarampión  \
     todos los países solo los 191 completos todos los países   
anio                                                            
2000             0.84                   0.84             0.69   
2005             0.82                   0.82             0.61   
2010             0.81                   0.81             0.56   
2015             0.81                   0.80             0.62   
2020             0.78                   0.78             0.57   
2024             0.76                   0.75             0.52   

                             
     solo los 189 completos  
anio                         
2000                   0.69  
2005                   0.61  
2010                   0.56  
2015                   0.61  
2020                   0.61  
2024                   0.56

Las correlaciones cambian a lo sumo 0,04, así que podemos mostrar todos los países disponibles en cada año sin que los huecos distorsionen la lectura.

**Valores extremos.** La esperanza de vida que publica Naciones Unidas incorpora las crisis de mortalidad (conflictos, epidemias), y algunos años quedan muy por debajo del resto de la serie del país:

In [19]:
panel.loc[panel["anio"].between(DESDE, HASTA) & (panel["vida"] < 35), ["pais", "anio", "vida", "agua", "sarampion"]]

,pais,anio,vida,agua,sarampion
1104,República Centroafricana,2009,14.665,48.701278,56.0
1114,República Centroafricana,2019,31.530,38.212266,37.0
1117,República Centroafricana,2022,18.818,36.295044,32.0
6146,Somalia,2011,32.453,45.541087,46.0


Son datos oficiales y no los borramos. Para que no aplasten la escala de todos los años, en los gráficos el eje vertical va de 35 a 88 años y estos puntos se dibujan sobre el borde inferior como círculos vacíos. La ficha muestra su valor real.

### La concentración cerca del 100 %

In [20]:
agua_ultimo = panel[(panel["anio"] == HASTA) & panel["agua"].notna() & panel["vida"].notna()]
print(f"{len(agua_ultimo)} países en {HASTA}")
for umbral in [90, 95, 99]:
    print(f"  con {umbral} % o más de acceso básico: {(agua_ultimo['agua'] >= umbral).sum()} ({(agua_ultimo['agua'] >= umbral).mean():.0%})")
altos = agua_ultimo[agua_ultimo["agua"] >= 95]
print(f"Esperanza de vida entre los de 95 % o más: de {altos['vida'].min():.1f} a {altos['vida'].max():.1f} años")

alt.Chart(agua_ultimo).mark_bar(color="#2a78d6").encode(
    x=alt.X("agua:Q", bin=alt.Bin(step=2.5, extent=[30, 102.5]), title="% de la población con acceso básico a agua para beber"),
    y=alt.Y("count():Q", title="Países"),
    tooltip=[alt.Tooltip("count():Q", title="Países")],
).properties(width=600, height=220, title=f"Países según el acceso básico a agua para beber, {HASTA}")


202 países en 2024
  con 90 % o más de acceso básico: 146 (72%)
  con 95 % o más de acceso básico: 125 (62%)
  con 99 % o más de acceso básico: 80 (40%)
Esperanza de vida entre los de 95 % o más: de 62.3 a 86.5 años


alt.Chart(...)

Más de la mitad de los países está en el último 5 % del eje, y entre ellos la esperanza de vida varía más de veinte años. En una escala lineal, todos esos puntos quedarían apilados en una franja angosta. Por eso el gráfico 2 **amplía el tramo de 90 % a 100 %**: ocupa casi la mitad del ancho y queda sombreado y rotulado para que no se lea como una escala uniforme.

La vacunación tiene otro problema. Los valores son enteros y muchos países comparten el mismo porcentaje, así que en una dispersión se tapan entre sí. Además, el umbral del 95 % es una referencia sanitaria. Por eso el gráfico 3 agrupa los países por tramos de cobertura y los muestra como un enjambre de puntos.

### El modelo del primer gráfico

El gráfico 1 compara la esperanza de vida de 2024 con la que «corresponde» al PIB por habitante. Ajustamos una regresión lineal de la esperanza de vida sobre el logaritmo del PIB per cápita: esperanza de vida = a + b · ln(PIB). La **brecha** de cada país es la diferencia entre su esperanza de vida observada y la que estima el modelo.

In [21]:
corte = panel[panel["anio"] == 2024].dropna(subset=["vida", "pib"]).copy()
b, a = np.polyfit(np.log(corte["pib"]), corte["vida"], 1)
corte["esperada"] = a + b * np.log(corte["pib"])
corte["brecha"] = corte["vida"] - corte["esperada"]
r2 = 1 - corte["brecha"].var() / corte["vida"].var()
print(f"{len(corte)} países · a = {a:.2f} · b = {b:.2f} · R² = {r2:.3f}")
print(f"Duplicar el PIB por habitante se asocia con {b * np.log(2):.1f} años más de esperanza de vida")
corte.sort_values("brecha")[["pais", "region", "pib", "vida", "esperada", "brecha"]].round(1).iloc[[0, 1, 2, -3, -2, -1]]


195 países · a = 22.85 · b = 5.26 · R² = 0.744
Duplicar el PIB por habitante se asocia con 3.6 años más de esperanza de vida


,pais,region,pib,vida,esperada,brecha
5004,Nigeria,África subsahariana,7994.0,54.6,70.1,-15.4
2869,Guyana,América Latina y el Caribe,70516.7,70.3,81.5,-11.2
5179,Nauru,Asia oriental y Pacífico,11957.9,62.3,72.2,-9.9
3814,Líbano,Medio Oriente y norte de África,10469.2,77.9,71.5,6.5
3429,Jordania,Medio Oriente y norte de África,10468.3,78.0,71.5,6.5
6019,Islas Salomón,Asia oriental y Pacífico,2353.5,70.7,63.7,7.0


¿El agua o la vacunación agregan algo al ingreso? Correlacionamos la brecha de cada país con ambos indicadores en 2024:

In [22]:
cruce = corte.dropna(subset=["agua", "sarampion"])
print(f"{len(cruce)} países con los tres datos")
for col in ["agua", "sarampion"]:
    print(f"{col:10} correlación con ln(PIB): {cruce[col].corr(np.log(cruce['pib'])):.2f} · con la brecha: {cruce[col].corr(cruce['brecha']):.2f}")


175 países con los tres datos
agua       correlación con ln(PIB): 0.79 · con la brecha: 0.15
sarampion  correlación con ln(PIB): 0.52 · con la brecha: 0.17


El acceso al agua está muy ligado al ingreso, y una vez que se tiene en cuenta el PIB casi no se relaciona con la brecha. La vacunación está menos atada al ingreso, pero tampoco explica mucho de quién vive más de lo esperado. Lo tenemos en cuenta al leer los gráficos 2 y 3: muestran asociaciones que en buena parte comparten con la riqueza.

# 4. Visualización 1: ¿Los países con mayor ingreso viven más?

En 2024, la esperanza de vida al nacer era de 84 años en Japón y de 54,6 en Nigeria. Una parte grande de esa distancia acompaña al ingreso: los países con mayor PIB por habitante suelen vivir más. El primer gráfico muestra esa regla y, sobre todo, a los países que se apartan de ella.

**Por qué este gráfico.** Una dispersión con el PIB en escala logarítmica muestra la forma de la relación, porque cada duplicación del ingreso ocupa la misma distancia, y deja ver cada país. La línea es la esperanza de vida que estima el modelo para cada nivel de ingreso. El color codifica la brecha en cinco clases divergentes: azul por encima de lo esperado, rojo por debajo y amarillo cerca de la línea. El área de cada círculo es proporcional a la población.

**Cómo usarlo** (los controles están debajo del gráfico):
- **Vista:** muestra todos los países o solo los que viven por encima, cerca o por debajo de lo esperado.
- **Región:** filtra por región del Banco Mundial.
- **Buscar un país:** al escribir su nombre (o parte), el país queda resaltado y rotulado.
- Al pasar el mouse sobre un círculo aparece su **ficha**: brecha, esperanza de vida observada y esperada, PIB por habitante y población.

In [23]:
CLASES = ["4 años o más por encima", "1,5 a 4 por encima", "Cerca de lo esperado", "1,5 a 4 por debajo", "4 años o más por debajo"]
COLORES_BRECHA = ["#194b87", "#397ba7", "#c9b24f", "#d0784f", "#9a2935"]

def clase(brecha):
    if brecha >= 4: return CLASES[0]
    if brecha >= 1.5: return CLASES[1]
    if brecha > -1.5: return CLASES[2]
    if brecha > -4: return CLASES[3]
    return CLASES[4]

g1 = corte[["iso3", "pais", "region", "poblacion", "pib", "vida", "esperada", "brecha"]].copy()
g1["clase"] = g1["brecha"].map(clase)
g1["poblacion_millones"] = g1["poblacion"] / 1e6
g1 = g1.sort_values("poblacion", ascending=False)      # los círculos grandes abajo, los chicos arriba

curva = pd.DataFrame({"pib": np.exp(np.linspace(np.log(700), np.log(150_000), 120))})
curva["vida"] = a + b * np.log(curva["pib"])

# Países que se rotulan siempre, a la derecha o a la izquierda del círculo
ROTULOS_DER = ["Japón", "Costa Rica", "Estados Unidos", "Nigeria", "Rusia", "India", "Sudáfrica", "Argentina", "Guinea Ecuatorial"]
ROTULOS_IZQ = ["Chile", "Sri Lanka"]


In [24]:
vista = alt.param(name="vista", value="Todos", bind=alt.binding_radio(
    options=["Todos", "Por encima", "Cerca de lo esperado", "Por debajo"], name="Vista: "))
region = alt.param(name="region", value="Todas", bind=alt.binding_select(
    options=["Todas"] + sorted(g1["region"].unique()), name="Región: "))
buscar = alt.param(name="buscar", value="", bind=alt.binding(input="search", placeholder="Ej.: Argentina", name="Buscar un país: "))

FILTRO_VISTA = ("vista == 'Todos' || (vista == 'Por encima' && datum.brecha >= 1.5) || "
                "(vista == 'Cerca de lo esperado' && abs(datum.brecha) < 1.5) || (vista == 'Por debajo' && datum.brecha <= -1.5)")
FILTRO_REGION = "region == 'Todas' || datum.region == region"
HAY_BUSQUEDA = "length(buscar) > 1"
COINCIDE = "length(buscar) > 1 && indexof(lower(datum.pais), lower(buscar)) >= 0"

x_pib = alt.X("pib:Q", title="PIB por habitante (PPA, $ internacionales de 2021, escala logarítmica)",
              scale=alt.Scale(type="log", domain=[700, 150_000]),
              axis=alt.Axis(values=[1000, 2000, 5000, 10_000, 20_000, 50_000, 100_000],
                            labelExpr="datum.value >= 1000 ? datum.value / 1000 + ' mil' : datum.value"))
y_vida = alt.Y("vida:Q", title="Esperanza de vida (años)", scale=alt.Scale(domain=[50, 88]))

linea = alt.Chart(curva).mark_line(color="#52514e", strokeWidth=2).encode(x=x_pib, y=y_vida)

puntos = alt.Chart(g1).mark_circle(stroke="white", strokeWidth=0.6).encode(
    x=x_pib, y=y_vida,
    size=alt.Size("poblacion:Q", scale=alt.Scale(domain=[0, g1["poblacion"].max()], range=[4, 3600]), legend=None),
    color=alt.Color("clase:N", title="Brecha con el modelo", scale=alt.Scale(domain=CLASES, range=COLORES_BRECHA),
                    legend=alt.Legend(orient="top", columns=3)),
    opacity=alt.condition(f"!({HAY_BUSQUEDA}) || {COINCIDE}", alt.value(0.85), alt.value(0.12)),
    tooltip=[
        alt.Tooltip("pais:N", title="País"),
        alt.Tooltip("region:N", title="Región"),
        alt.Tooltip("brecha:Q", title="Brecha con el modelo (años)", format="+.1f"),
        alt.Tooltip("vida:Q", title="Esperanza de vida (años)", format=".1f"),
        alt.Tooltip("esperada:Q", title="Esperada según el PIB (años)", format=".1f"),
        alt.Tooltip("pib:Q", title="PIB por habitante (PPA, $)", format=",.0f"),
        alt.Tooltip("poblacion_millones:Q", title="Población (millones)", format=",.1f"),
    ],
).transform_filter(FILTRO_VISTA).transform_filter(FILTRO_REGION)

def rotulos(nombres, alinear, dx):
    condicion = f"({HAY_BUSQUEDA}) ? ({COINCIDE}) : indexof({nombres!r}, datum.pais) >= 0"
    return alt.Chart(g1).mark_text(align=alinear, dx=dx, fontSize=12, fontWeight="bold").encode(
        x=x_pib, y=y_vida, text="pais:N",
    ).transform_filter(FILTRO_VISTA).transform_filter(FILTRO_REGION).transform_filter(
        condicion if alinear == "left" else f"!({HAY_BUSQUEDA}) && indexof({nombres!r}, datum.pais) >= 0")

grafico_1 = (linea + puntos + rotulos(ROTULOS_DER, "left", 9) + rotulos(ROTULOS_IZQ, "right", -9)).add_params(
    vista, region, buscar
).properties(
    width=820, height=500,
    title={"text": "¿Los países con mayor ingreso viven más?",
           "subtitle": ["Esperanza de vida al nacer y PIB por habitante, 2024. La línea marca la esperanza de vida que estima el modelo según el ingreso;",
                        "el color indica cuánto se aleja cada país de ella y el área del círculo, su población. Fuente: Banco Mundial (WDI), vía Data360."]},
)
grafico_1


alt.LayerChart(...)

In [25]:
print(f"Duplicar el PIB por habitante: +{b * np.log(2):.1f} años · R² = {r2:.1%} · {len(g1)} países")
grandes = g1[g1["poblacion"] >= 1_000_000]
print("\nMayores brechas positivas (países de al menos 1 millón de habitantes):")
display(grandes.nlargest(6, "brecha")[["pais", "region", "pib", "vida", "brecha"]].round(1))
print("Mayores brechas negativas:")
display(grandes.nsmallest(6, "brecha")[["pais", "region", "pib", "vida", "brecha"]].round(1))
g1[g1["iso3"].isin(["USA", "CRI", "ARG", "CHL"])][["pais", "pib", "vida", "esperada", "brecha"]].round(1)


Duplicar el PIB por habitante: +3.6 años · R² = 74.4% · 195 países

Mayores brechas positivas (países de al menos 1 millón de habitantes):


,pais,region,pib,vida,brecha
3429,Jordania,Medio Oriente y norte de África,10468.3,78.0,6.5
3814,Líbano,Medio Oriente y norte de África,10469.2,77.9,6.5
4829,Malaui,África subsahariana,1634.2,67.6,5.8
5039,Nicaragua,América Latina y el Caribe,7661.1,75.1,5.2
3744,Kuwait,Medio Oriente y norte de África,46664.2,84.6,5.2
3989,Sri Lanka,Asia meridional,13793.1,77.7,4.7


Mayores brechas negativas:


,pais,region,pib,vida,brecha
5004,Nigeria,África subsahariana,7994.0,54.6,-15.4
2659,Guinea Ecuatorial,África subsahariana,14997.8,63.9,-9.5
6614,Chad,África subsahariana,2430.7,55.2,-8.6
6439,Esuatini,África subsahariana,10386.2,64.3,-7.2
1329,Côte d’Ivoire,África subsahariana,6747.0,62.1,-7.1
4024,Lesoto,África subsahariana,2697.6,57.8,-6.6


,pais,pib,vida,esperada,brecha
7139,Estados Unidos,75698.2,78.9,81.9,-3.0
244,Argentina,26772.0,77.5,76.4,1.1
1259,Chile,30267.6,81.4,77.1,4.3
1574,Costa Rica,27325.0,81.0,76.5,4.5


**Lectura.** La regla se cumple. El modelo con el PIB per cápita da cuenta de cerca de tres cuartas partes de la variación de la esperanza de vida entre países, y a lo largo de la línea cada duplicación del ingreso se asocia con unos 3,6 años más. Pero la nube es ancha. Costa Rica vive 2,1 años más que Estados Unidos con un tercio de su ingreso, y Chile también lo supera. En el otro extremo, Nigeria queda unos 15 años por debajo de lo que estima el modelo, y casi todas las brechas negativas más grandes corresponden a África subsahariana. Argentina queda cerca de la línea.

La brecha es una distancia a un promedio, no una nota: dice qué países se apartan del patrón, pero no por qué. Para buscar pistas miramos dos factores que no dependen solo de la riqueza, el agua y las vacunas.

# 5. Visualización 2: Agua para beber y esperanza de vida

Tener una fuente de agua mejorada cerca de casa reduce las enfermedades diarreicas, una de las principales causas de muerte infantil en los países pobres. ¿Viven más los países donde más gente tiene acceso básico al agua? ¿Cambió esa relación entre 2000 y 2024?

**Por qué este gráfico.** Para relacionar dos variables continuas país por país, la dispersión es la forma más directa. El deslizador de años muestra cómo se mueve la nube sin cambiar las escalas, así que cualquier desplazamiento es un cambio en los datos. Como más de la mitad de los países está por encima del 95 %, el tramo de 90 % a 100 % se amplía (ver la sección 3). Todos los puntos tienen el mismo tamaño porque aquí cada país es una observación de la relación. El color indica el grupo de ingreso, para conectar con el gráfico 1.

**Cómo usarlo** (los controles están debajo del gráfico):
- Mové el **deslizador de año**. Arriba a la izquierda se indica cuántos países aparecen en ese año, junto con un resumen de la relación.
- Al pasar el mouse sobre un punto aparece su **ficha**: país, año, acceso al agua y esperanza de vida.
- **Buscar un país** lo resalta y dibuja su recorrido desde 2000 hasta el año elegido.

In [26]:
INGRESO_ORDEN = ["Bajo", "Medio bajo", "Medio alto", "Alto", "Sin clasificar"]
INGRESO_COLORES = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b", "#b4b2a9"]
color_ingreso = alt.Color("ingreso:N", title="Ingreso del país", scale=alt.Scale(domain=INGRESO_ORDEN, range=INGRESO_COLORES),
                          legend=alt.Legend(orient="right"))
VIDA_MIN, VIDA_MAX = 35, 88

def datos_anuales(col, decimales):
    d = panel[panel["anio"].between(DESDE, HASTA) & panel[col].notna() & panel["vida"].notna()]
    return d.assign(valor=d[col].round(decimales), vida=d["vida"].round(1))[["iso3", "pais", "region", "ingreso", "anio", "valor", "vida"]]


def resumen_anual(d, cortes, etiquetas):
    grupos = pd.cut(d["valor"], cortes, labels=etiquetas, right=False)
    tabla = d.groupby(["anio", grupos], observed=False)["vida"].agg(["size", "median"]).unstack()
    tabla.columns = [f"{'países' if m == 'size' else 'mediana vida'} {g}" for m, g in tabla.columns]
    tabla.insert(0, "correlación", d.groupby("anio")[["valor", "vida"]].corr().xs("valor", level=1)["vida"])
    tabla.insert(0, "países", d.groupby("anio").size())
    return tabla.round(2)


agua = datos_anuales("agua", 1)
resumen_agua = resumen_anual(agua, [0, 80, 95, 101], ["< 80 %", "80–95 %", "≥ 95 %"])
resumen_agua.loc[[2000, 2005, 2010, 2015, 2020, 2024]]


,países,correlación,países < 80 %,países 80–95 %,países ≥ 95 %,mediana vida < 80 %,mediana vida 80–95 %,mediana vida ≥ 95 %
anio,,,,,,,,
2000,203,0.84,60,57,86,54.80,70.3,75.35
2005,209,0.82,55,56,98,56.80,70.8,76.40
2010,212,0.81,53,53,106,59.70,70.3,76.75
2015,213,0.81,51,48,114,62.30,71.2,77.30
2020,205,0.78,42,47,116,61.65,70.2,77.40
2024,202,0.76,37,40,125,63.80,71.3,78.30


In [27]:
def coma(v, decimales=1):
    return f"{v:.{decimales}f}".replace(".", ",")

# Texto que acompaña a cada año: cantidad de países y resumen de la relación
textos_agua = pd.DataFrame([{
    "anio": anio,
    "linea_1": f"{int(fila['países'])} países con ambos datos en {anio}",
    "linea_2": f"Correlación: {coma(fila['correlación'], 2)} · mediana de esperanza de vida: {coma(fila['mediana vida < 80 %'])} años "
               f"con menos del 80 % de acceso y {coma(fila['mediana vida ≥ 95 %'])} con 95 % o más",
} for anio, fila in resumen_agua.iterrows()])

ANCHO, ALTO = 760, 460
anio_agua = alt.param(name="anio_agua", value=HASTA, bind=alt.binding_range(min=DESDE, max=HASTA, step=1, name="Año: "))
buscar_agua = alt.param(name="buscar_agua", value="", bind=alt.binding(input="search", placeholder="Ej.: Argentina", name="Buscar un país: "))
COINCIDE_AGUA = "length(buscar_agua) > 1 && indexof(lower(datum.pais), lower(buscar_agua)) >= 0"

# Escala por tramos: de 15 % a 90 % ocupa el 55 % del ancho y de 90 % a 100 %, el 45 % restante
escala_agua = alt.Scale(domain=[15, 90, 100], range=[0, ANCHO * 0.55, ANCHO], nice=False)
eje_agua = alt.Axis(values=[20, 40, 60, 80, 90, 92, 94, 96, 98, 100], labelExpr="datum.value + ' %'")
TITULO_X_AGUA = "Población que usa al menos un servicio básico de agua para beber (%)"
escala_vida = alt.Scale(domain=[VIDA_MIN, VIDA_MAX], clamp=True)
x_agua = alt.X("valor:Q", title=TITULO_X_AGUA, scale=escala_agua, axis=eje_agua)
y_agua = alt.Y("vida:Q", title="Esperanza de vida (años)", scale=escala_vida)

tramo = alt.Chart(pd.DataFrame({"desde": [90], "hasta": [100]})).mark_rect(color="#e1e0d9", opacity=0.45).encode(
    x=alt.X("desde:Q", title=TITULO_X_AGUA, scale=escala_agua, axis=eje_agua), x2="hasta:Q")
corte_tramo = alt.Chart(pd.DataFrame({"valor": [90]})).mark_rule(strokeDash=[3, 3], color="#898781").encode(x=x_agua)
rotulo_tramo = alt.Chart(pd.DataFrame({"valor": [90.3], "vida": [VIDA_MAX - 1]})).mark_text(
    align="left", baseline="top", fontSize=11, color="#898781", text="Tramo ampliado: 90 % a 100 %").encode(x=x_agua, y=y_agua)

base_agua = alt.Chart(agua).transform_filter("datum.anio == anio_agua")
FICHA_AGUA = [
        alt.Tooltip("pais:N", title="País"),
        alt.Tooltip("anio:Q", title="Año", format="d"),
        alt.Tooltip("valor:Q", title="Acceso básico a agua (% de la población)", format=".1f"),
        alt.Tooltip("vida:Q", title="Esperanza de vida (años)", format=".1f"),
        alt.Tooltip("ingreso:N", title="Grupo de ingreso"),
]
puntos_agua = base_agua.transform_filter(f"datum.vida >= {VIDA_MIN}").mark_circle(size=60, stroke="white", strokeWidth=0.5).encode(
    x=x_agua, y=y_agua, color=color_ingreso,
    opacity=alt.condition(f"length(buscar_agua) < 2 || {COINCIDE_AGUA}", alt.value(0.85), alt.value(0.15)),
    tooltip=FICHA_AGUA,
)
# Esperanza de vida menor de 35 años: círculo vacío sobre el borde inferior
fuera_agua = base_agua.transform_filter(f"datum.vida < {VIDA_MIN}").mark_point(size=60, filled=False, strokeWidth=1.5).encode(
    x=x_agua, y=y_agua, color=color_ingreso, tooltip=FICHA_AGUA)
# Recorrido del país buscado desde 2000 hasta el año elegido
recorrido = alt.Chart(agua).transform_filter(f"{COINCIDE_AGUA} && datum.anio <= anio_agua")
linea_recorrido = recorrido.mark_line(color="#52514e", strokeWidth=1.5, point=alt.OverlayMarkDef(size=12, color="#52514e")).encode(
    x=x_agua, y=y_agua, detail="pais:N", order="anio:Q")
nombre_recorrido = base_agua.transform_filter(COINCIDE_AGUA).mark_text(align="left", dx=8, dy=-8, fontWeight="bold").encode(
    x=x_agua, y=y_agua, text="pais:N")

encabezado_agua = alt.Chart(textos_agua).transform_filter("datum.anio == anio_agua")
linea_1_agua = encabezado_agua.mark_text(align="left", x=8, y=-28, fontSize=13, fontWeight="bold").encode(text="linea_1:N")
linea_2_agua = encabezado_agua.mark_text(align="left", x=8, y=-12, fontSize=11.5, color="#52514e").encode(text="linea_2:N")

grafico_2 = (tramo + corte_tramo + rotulo_tramo + linea_recorrido + puntos_agua + fuera_agua + nombre_recorrido + linea_1_agua + linea_2_agua).add_params(
    anio_agua, buscar_agua
).properties(
    width=ANCHO, height=ALTO,
    title={"text": "¿Viven más los países donde más gente tiene agua para beber?",
           "subtitle": ["Cada punto es un país. Las escalas quedan fijas al mover el año y solo aparecen los países con ambos datos.",
                        "Los círculos vacíos sobre el borde inferior tienen una esperanza de vida menor de 35 años. Fuente: Banco Mundial (WDI), vía Data360."],
           "offset": 40},
)
grafico_2


alt.LayerChart(...)

In [28]:
cambios = agua[agua["anio"].isin([DESDE, HASTA])].pivot(index="pais", columns="anio", values=["valor", "vida"]).dropna()
cambios = pd.DataFrame({"agua_2000": cambios[("valor", DESDE)], "agua_2024": cambios[("valor", HASTA)],
                        "vida_2000": cambios[("vida", DESDE)], "vida_2024": cambios[("vida", HASTA)]})
cambios["más agua"] = cambios["agua_2024"] - cambios["agua_2000"]
cambios["más años"] = cambios["vida_2024"] - cambios["vida_2000"]
print(f"{len(cambios)} países con datos en {DESDE} y {HASTA}")
print(f"Correlación entre lo que ganó cada país en acceso al agua y en esperanza de vida: {cambios['más agua'].corr(cambios['más años']):.2f}")
display(cambios.nlargest(6, "más agua").round(1))
print(f"Esperanza de vida más baja entre los países con 95 % o más de acceso en {HASTA}:")
agua[(agua["anio"] == HASTA) & (agua["valor"] >= 95)].nsmallest(5, "vida")[["pais", "valor", "vida", "ingreso"]]


191 países con datos en 2000 y 2024
Correlación entre lo que ganó cada país en acceso al agua y en esperanza de vida: 0.51


,agua_2000,agua_2024,vida_2000,vida_2024,más agua,más años
pais,,,,,,
Afganistán,29.7,80.8,55.0,66.3,51.1,11.3
Somalia,26.6,75.3,50.7,59.0,48.7,8.3
Laos,48.3,95.9,58.3,69.2,47.6,10.9
Mozambique,23.4,66.6,49.5,63.8,43.2,14.3
Myanmar (Birmania),47.0,85.7,60.4,67.1,38.7,6.7
Nigeria,44.5,82.2,47.1,54.6,37.7,7.5


Esperanza de vida más baja entre los países con 95 % o más de acceso en 2024:


,pais,valor,vida,ingreso
5179,Nauru,97.8,62.3,Alto
6964,Tuvalu,98.7,67.3,Medio alto
2239,Fiyi,95.6,67.5,Medio alto
3779,Laos,95.9,69.2,Medio bajo
5669,Territorios Palestinos,98.8,69.2,Medio bajo


**Lectura.** La relación es fuerte en todo el período, aunque se debilita: la correlación baja de 0,84 en 2000 a 0,76 en 2024. En 2000, la mediana de la esperanza de vida era de 54,8 años en los 60 países donde menos del 80 % de la población tenía acceso básico, y de unos 75 años en los que llegaban al 95 % o más: más de 20 años de diferencia. En 2024 los países por debajo del 80 % eran 37, y la diferencia de medianas se redujo a 14,5 años (63,8 contra 78,3).

Al recorrer los años, la nube se desplaza hacia arriba y a la derecha. Afganistán, Somalia, Laos y Mozambique sumaron más de 40 puntos de acceso entre 2000 y 2024, y ganaron entre 8 y 14 años de esperanza de vida. Pero en los 191 países con datos en ambos años, la correlación entre lo que ganaron en acceso y en años de vida es de 0,51: positiva, pero lejos de ser una regla. El tramo ampliado muestra la otra mitad de la historia. Entre los países con 95 % o más de acceso, la esperanza de vida va de 62 a 86 años. Llegar al agua básica acompaña a vivir más, pero no alcanza: Nauru, Tuvalu y Fiyi superan el 95 % de acceso y viven menos de 68 años.

**Asociación no es causalidad.** El acceso al agua avanza junto con el ingreso, la urbanización, el saneamiento y los servicios de salud. En 2024, su correlación con el logaritmo del PIB fue de 0,79, y una vez descontado el ingreso casi no se relaciona con la brecha del gráfico 1. El gráfico muestra que el agua y la esperanza de vida van juntas, no cuánto de la esperanza de vida se debe al agua.

# 6. Visualización 3: Vacunación contra el sarampión y esperanza de vida

El agua es infraestructura; la vacunación es prevención. Es un programa relativamente barato que depende de que el sistema de salud llegue a cada chico. ¿Los países que vacunan más contra el sarampión viven más? (En la sección 3 explicamos por qué elegimos este indicador en lugar de la secundaria.)

**Por qué este gráfico.** La cobertura se publica en números enteros y muchos países comparten el mismo valor, así que en una dispersión se superponen. Además, el 95 % es una meta sanitaria con sentido propio. Por eso agrupamos los países en cuatro tramos de cobertura (menos de 80 %, 80–89 %, 90–94 % y 95 % o más) y los mostramos en un **enjambre**: cada punto es un país, separado de sus vecinos para que no se tapen. La línea horizontal marca la mediana de cada grupo y arriba se indica cuántos países hay en cada uno. Los tramos y las escalas no cambian al mover el año.

**Cómo usarlo** (los controles están debajo del gráfico):
- Mové el **deslizador de año**. Los países cambian de grupo cuando cambia su cobertura.
- La **ficha** de cada punto muestra el país, el año, la cobertura, su grupo y la esperanza de vida.
- **Buscar un país** lo deja resaltado mientras se recorren los años.

In [29]:
GRUPOS = ["Menos de 80 %", "80 % a 89 %", "90 % a 94 %", "95 % o más"]
vacunas = datos_anuales("sarampion", 0)
vacunas["grupo_n"] = pd.cut(vacunas["valor"], [0, 80, 90, 95, 101], labels=False, right=False).astype(int)
vacunas["grupo"] = vacunas["grupo_n"].map(dict(enumerate(GRUPOS)))
resumen_vacunas = resumen_anual(vacunas, [0, 80, 90, 95, 101], ["< 80 %", "80–89 %", "90–94 %", "≥ 95 %"])
resumen_vacunas.loc[[2000, 2005, 2010, 2015, 2019, 2020, 2021, 2022, 2024]]


,países,correlación,países < 80 %,países 80–89 %,países 90–94 %,países ≥ 95 %,mediana vida < 80 %,mediana vida 80–89 %,mediana vida 90–94 %,mediana vida ≥ 95 %
anio,,,,,,,,,,
2000,190,0.69,63,41,32,54,56.60,70.90,72.55,72.65
2005,191,0.61,51,33,34,73,58.50,66.50,73.15,73.00
2010,192,0.56,41,30,35,86,60.00,69.25,72.90,74.15
2015,193,0.62,41,37,32,83,63.40,69.40,76.50,74.90
2019,193,0.60,39,34,37,83,63.50,70.20,75.00,75.90
2020,193,0.57,52,40,43,58,65.45,69.95,74.80,77.05
2021,193,0.55,59,40,35,59,66.20,70.15,75.40,74.40
2022,193,0.57,52,43,32,66,66.50,71.60,74.90,76.75
2024,192,0.52,45,45,35,67,66.50,71.30,76.50,76.80


Para el enjambre calculamos, para cada año y grupo, un desplazamiento horizontal que evita que los puntos se superpongan. Recorremos los países de menor a mayor esperanza de vida y ubicamos cada uno en la posición libre más cercana al centro de su columna.

In [30]:
ANCHO_3, ALTO_3, RADIO = 760, 440, 4.2
px_por_anio = ALTO_3 / (VIDA_MAX - VIDA_MIN)          # píxeles por año de esperanza de vida
px_por_grupo = ANCHO_3 / len(GRUPOS)                   # cada grupo ocupa una unidad del eje x

def enjambre(grupo):
    alto = grupo["vida"].clip(lower=VIDA_MIN) * px_por_anio
    colocados, desplazamientos = [], {}
    paso = RADIO * 0.9
    for i in alto.sort_values(kind="stable").index:
        y = alto[i]
        vecinos = [(dx, yy) for dx, yy in colocados if abs(yy - y) < 2 * RADIO + 1]
        for k in range(400):
            dx = 0 if k == 0 else (1 if k % 2 else -1) * ((k + 1) // 2) * paso
            if all((dx - vx) ** 2 + (y - vy) ** 2 >= (2 * RADIO + 0.6) ** 2 for vx, vy in vecinos):
                break
        dx = max(-0.45 * px_por_grupo, min(0.45 * px_por_grupo, dx))
        colocados.append((dx, y))
        desplazamientos[i] = dx / px_por_grupo
    return pd.Series(desplazamientos)

vacunas = vacunas.sort_values(["anio", "grupo_n", "iso3"])
vacunas["x"] = vacunas["grupo_n"] + vacunas.groupby(["anio", "grupo_n"], group_keys=False)[["vida"]].apply(enjambre)

medianas = vacunas.groupby(["anio", "grupo_n"])["vida"].agg(mediana="median", n="size").reset_index()
medianas["desde"], medianas["hasta"] = medianas["grupo_n"] - 0.38, medianas["grupo_n"] + 0.38
medianas["texto_mediana"] = medianas["mediana"].map(coma)
medianas["texto_n"] = medianas["n"].astype(str) + " países"

textos_vacunas = pd.DataFrame([{
    "anio": anio,
    "linea_1": f"{int(fila['países'])} países con ambos datos en {anio}",
    "linea_2": f"Correlación entre cobertura y esperanza de vida: {coma(fila['correlación'], 2)}",
} for anio, fila in resumen_vacunas.iterrows()])
vacunas[["anio", "pais", "valor", "grupo", "vida", "x"]].head()


,anio,pais,valor,grupo,vida,x
45,2000,Afganistán,27.0,Menos de 80 %,55.0,0.059684
80,2000,Angola,41.0,Menos de 80 %,46.5,-0.059684
395,2000,Austria,75.0,Menos de 80 %,78.1,0.000000
430,2000,Azerbaiyán,67.0,Menos de 80 %,65.4,0.000000
465,2000,Burundi,72.0,Menos de 80 %,47.6,0.059684


In [31]:
anio_vac = alt.param(name="anio_vac", value=HASTA, bind=alt.binding_range(min=DESDE, max=HASTA, step=1, name="Año: "))
buscar_vac = alt.param(name="buscar_vac", value="", bind=alt.binding(input="search", placeholder="Ej.: Argentina", name="Buscar un país: "))
COINCIDE_VAC = "length(buscar_vac) > 1 && indexof(lower(datum.pais), lower(buscar_vac)) >= 0"

escala_x3 = alt.Scale(domain=[-0.5, 3.5], nice=False, zero=False)
eje_x3 = alt.Axis(values=[0, 1, 2, 3], grid=False, ticks=False, domain=False, labelFontSize=12, labelPadding=8,
                  labelExpr=f"{GRUPOS!r}[datum.value]")
TITULO_X3 = "Cobertura de la primera dosis contra el sarampión (% de niños de 12 a 23 meses)"
x3 = alt.X("x:Q", title=TITULO_X3, scale=escala_x3, axis=eje_x3)
y3 = alt.Y("vida:Q", title="Esperanza de vida (años)", scale=escala_vida)

base_vac = alt.Chart(vacunas).transform_filter("datum.anio == anio_vac")
FICHA_VAC = [
        alt.Tooltip("pais:N", title="País"),
        alt.Tooltip("anio:Q", title="Año", format="d"),
        alt.Tooltip("valor:Q", title="Vacunación contra el sarampión (%)", format="d"),
        alt.Tooltip("grupo:N", title="Grupo"),
        alt.Tooltip("vida:Q", title="Esperanza de vida (años)", format=".1f"),
        alt.Tooltip("ingreso:N", title="Grupo de ingreso"),
]
puntos_vac = base_vac.transform_filter(f"datum.vida >= {VIDA_MIN}").mark_circle(size=50, stroke="white", strokeWidth=0.5).encode(
    x=x3, y=y3, color=color_ingreso,
    opacity=alt.condition(f"length(buscar_vac) < 2 || {COINCIDE_VAC}", alt.value(0.9), alt.value(0.15)),
    tooltip=FICHA_VAC,
)
fuera_vac = base_vac.transform_filter(f"datum.vida < {VIDA_MIN}").mark_point(size=50, filled=False, strokeWidth=1.5).encode(
    x=x3, y=y3, color=color_ingreso, tooltip=FICHA_VAC)
nombre_vac = alt.Chart(vacunas).transform_filter(f"datum.anio == anio_vac && {COINCIDE_VAC}").mark_text(
    align="left", dx=7, dy=-7, fontWeight="bold").encode(x=x3, y=y3, text="pais:N")

base_med = alt.Chart(medianas).transform_filter("datum.anio == anio_vac")
x_med = lambda campo: alt.X(campo, title=TITULO_X3, scale=escala_x3, axis=eje_x3)
y_med = alt.Y("mediana:Q", title="Esperanza de vida (años)", scale=escala_vida)
linea_med = base_med.mark_rule(strokeWidth=2.5, color="#0b0b0b").encode(x=x_med("desde:Q"), x2="hasta:Q", y=y_med)
texto_med = base_med.mark_text(align="left", dx=4, fontWeight="bold", fontSize=12).encode(x=x_med("hasta:Q"), y=y_med, text="texto_mediana:N")
texto_n = base_med.mark_text(fontSize=12, color="#52514e", y=-8).encode(x=x_med("grupo_n:Q"), text="texto_n:N")

encabezado_vac = alt.Chart(textos_vacunas).transform_filter("datum.anio == anio_vac")
linea_1_vac = encabezado_vac.mark_text(align="left", x=0, y=-44, fontSize=13, fontWeight="bold").encode(text="linea_1:N")
linea_2_vac = encabezado_vac.mark_text(align="left", x=0, y=-28, fontSize=11.5, color="#52514e").encode(text="linea_2:N")

grafico_3 = (puntos_vac + fuera_vac + linea_med + texto_med + texto_n + nombre_vac + linea_1_vac + linea_2_vac).add_params(
    anio_vac, buscar_vac
).properties(
    width=ANCHO_3, height=ALTO_3,
    title={"text": "¿Viven más los países que vacunan más contra el sarampión?",
           "subtitle": ["Cada punto es un país, ubicado según la cobertura de la primera dosis y su esperanza de vida. La línea marca la mediana de cada grupo.",
                        "Los círculos vacíos sobre el borde inferior tienen una esperanza de vida menor de 35 años. Fuente: Banco Mundial (WDI), vía Data360."],
           "offset": 56},
)
grafico_3


alt.LayerChart(...)

In [32]:
ultimo_vac = vacunas[vacunas["anio"] == HASTA]
print(f"Países con menos de 80 % de cobertura y esperanza de vida de 75 años o más en {HASTA}:")
display(ultimo_vac[(ultimo_vac["valor"] < 80) & (ultimo_vac["vida"] >= 75)].sort_values("valor")[["pais", "valor", "vida", "ingreso"]])
vacunas[vacunas["iso3"].isin(["ARG", "BRA", "MEX", "USA"])].pivot(index="anio", columns="pais", values="valor").loc[[2000, 2010, 2019, 2020, 2021, 2022, 2024]]


Países con menos de 80 % de cobertura y esperanza de vida de 75 años o más en 2024:


,pais,valor,vida,ingreso
4619,Montenegro,23.0,77.9,Medio alto
3814,Líbano,67.0,77.9,Medio bajo
769,Bosnia y Herzegovina,69.0,78.0,Medio alto
4479,Macedonia del Norte,73.0,76.6,Medio alto
1994,Ecuador,74.0,77.6,Medio alto
5774,Rumanía,78.0,76.5,Alto


pais,Argentina,Brasil,Estados Unidos,México
anio,,,,
2000,91.0,99.0,91.0,96.0
2010,95.0,99.0,92.0,95.0
2019,86.0,91.0,90.0,73.0
2020,77.0,79.0,91.0,92.0
2021,86.0,73.0,92.0,99.0
2022,85.0,81.0,92.0,86.0
2024,82.0,93.0,92.0,80.0


**Lectura.** Los países que vacunan a menos del 80 % de sus chicos viven bastante menos: en 2024, la mediana fue de 66,5 años, contra 76,8 en los que alcanzan el 95 %. Pero la relación es más débil que la del agua y también se debilita con el tiempo: la correlación baja de 0,69 en 2000 a 0,52 en 2024. El enjambre muestra por qué. Por encima del 90 %, las medianas casi no se distinguen, y en los grupos de menor cobertura aparecen países con esperanza de vida alta. Montenegro, Bosnia y Herzegovina, Macedonia del Norte, Ecuador y Rumanía viven más de 76 años, son de ingreso medio alto o alto y en 2024 vacunaron a menos del 80 % de sus chicos.

El deslizador también deja ver un retroceso. Entre 2019 y 2021, durante la pandemia, cuando se interrumpieron muchos servicios de vacunación, el grupo de 95 % o más pasó de 83 a 59 países, y en 2024 todavía no se había recuperado (67). En América Latina, Argentina bajó de 95 % en 2010 a 82 % en 2024, Brasil cayó a 73 % en 2021 y México tuvo 80 % en 2024.

**Asociación no es causalidad.** La esperanza de vida al nacer resume la mortalidad de todas las edades en un año, y la cobertura de vacunación de ese mismo año dice poco de ella. Una caída reciente puede tardar años en traducirse en brotes y muertes, y la vacunación suele avanzar junto con todo el sistema de salud. El gráfico muestra qué países vacunan menos de lo que su longevidad haría esperar, pero no cuántos años de vida aporta la vacuna.

# 7. Conclusiones

- **El ingreso es la regla, no el destino.** En 2024, el PIB por habitante da cuenta de cerca de tres cuartas partes de la variación de la esperanza de vida entre países. El cuarto restante es donde aparecen los casos interesantes: Costa Rica y Chile viven más que Estados Unidos con mucho menos ingreso, y Nigeria vive unos 15 años menos de lo que estima el modelo.
- **El agua acompaña a la longevidad, pero va de la mano del ingreso.** La relación entre acceso básico al agua y esperanza de vida es fuerte (correlación de 0,76 en 2024), y la distancia entre los países con y sin acceso se achicó desde 2000. Sin embargo, el agua está tan ligada al PIB que casi no ayuda a entender quién vive más de lo esperado, y entre los países con acceso casi universal la esperanza de vida varía más de veinte años.
- **La vacunación cuenta otra historia.** Su relación con la esperanza de vida es más débil y se fue debilitando. La caída de la cobertura durante la pandemia, y la baja cobertura de países de ingreso medio y alto que viven mucho, muestran que la vacunación ya no se reparte solo según la riqueza o el desarrollo.
- **Ninguno de estos gráficos prueba causas.** Todos muestran asociaciones entre países. Para estimar cuánto aporta cada factor harían falta otros datos (series largas por país, información dentro de cada país o diseños que comparen situaciones parecidas) y otro tipo de análisis.

### Fuentes

Banco Mundial, *World Development Indicators*, consultados con la [API de Data360](https://data360.worldbank.org/en/api) al ejecutar el notebook. Los datos de origen son:

- **Esperanza de vida:** estimaciones de población de Naciones Unidas.
- **Acceso al agua:** Programa Conjunto de Monitoreo OMS/UNICEF (JMP).
- **Vacunación:** estimaciones OMS/UNICEF de cobertura de inmunización (WUENIC).

Las regiones y los grupos de ingreso corresponden a la clasificación vigente del Banco Mundial y se aplican a todos los años.